# Fresh post-freeze holdout: end-to-end evaluation

I use this notebook once to evaluate the frozen dissertation models on observations after **17 July 2026**. It downloads the new SPY, SPX and VIX minutes, rebuilds the original features, loads the already-saved models, and reports RQ1-RQ5 without refitting anything.

> **Research-integrity warning:** the first successful run reveals the holdout outcomes. After that, these dates are no longer fresh. I do not tune models, thresholds, filters, costs, strikes or exit rules from the results here.

The primary evidence comes from the canonical classical models. The saved LSTM models are evaluated separately as an optional exploratory comparison. RQ5 is only calculated when the frozen RQ1-RQ3 rule selects a date and Massive returns usable option bars.


## 1. Run controls and frozen protocol

Before running all cells, I read the warning above and change `CONFIRM_FRESH_HOLDOUT` to the exact confirmation text. `END_DATE_OVERRIDE` can be used to stop at an earlier completed session, but it must not be moved around after looking at results.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import sys
import time
from datetime import time as clock_time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)

CONFIRM_FRESH_HOLDOUT = "YeeHaw"
REQUIRED_CONFIRMATION = "YeeHaw"  # This is a research-integrity warning. Do not change it without reading the dissertation.

# None uses the last completed US cash session at notebook run time.
END_DATE_OVERRIDE = None
DOWNLOAD_UNDERLYINGS = True
DOWNLOAD_OPTIONS_FOR_RQ5 = True
RUN_EXPLORATORY_LSTM = True
FORCE_REDOWNLOAD = False
ALLOW_OVERWRITE_FINALIZED_RUN = False

if CONFIRM_FRESH_HOLDOUT != REQUIRED_CONFIRMATION:
    raise RuntimeError(
        "Read the research-integrity warning, then set CONFIRM_FRESH_HOLDOUT "
        f"to {REQUIRED_CONFIRMATION!r}."
    )


def locate_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "config.yaml").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not locate the dissertation project root.")


PROJECT_ROOT = locate_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.massive_database import MassiveREST, normalize_aggregates

HOLDOUT_START_DATE = pd.Timestamp("2026-07-18")
LAST_TRAINING_DATE = pd.Timestamp("2026-03-20")
LAST_PREVIOUSLY_VIEWED_DATE = pd.Timestamp("2026-07-17")
LARGE_MOVE_THRESHOLD_BPS = 27.504041884352514

# Frozen from the 70th percentile of RQ1 development confidence_raw.
RQ1_HIGH_CONFIDENCE_DISTANCE = 0.205260582211788
RQ2_MIN_PREDICTED_MOVE_BPS = 20.0
RQ3_REQUIRED = True

RANDOM_STATE = 42
TIMEZONE = "America/New_York"
OPTION_OFFSETS = [0, 5, 10, 15, 20, 25, 30]
CONTRACT_MULTIPLIER = 100.0

now_et = pd.Timestamp.now(tz=TIMEZONE)
if END_DATE_OVERRIDE is not None:
    HOLDOUT_END_DATE = pd.Timestamp(END_DATE_OVERRIDE).normalize()
elif now_et.time() >= clock_time(16, 10):
    HOLDOUT_END_DATE = now_et.tz_localize(None).normalize()
else:
    HOLDOUT_END_DATE = (now_et.tz_localize(None).normalize() - pd.offsets.BDay(1))

if HOLDOUT_END_DATE < HOLDOUT_START_DATE:
    raise RuntimeError("No post-freeze sessions are available for the requested period.")

RUN_ID = f"{HOLDOUT_START_DATE:%Y%m%d}_{HOLDOUT_END_DATE:%Y%m%d}"
HOLDOUT_DATA_ROOT = PROJECT_ROOT / "data" / "fresh_holdout_post_2026_07_17"
HOLDOUT_ROOT = PROJECT_ROOT / "outputs" / "fresh_holdout_post_2026_07_17" / RUN_ID
TABLE_ROOT = HOLDOUT_ROOT / "tables"
FIGURE_ROOT = HOLDOUT_ROOT / "figures"
OPTION_ROOT = HOLDOUT_ROOT / "options"
for path in [HOLDOUT_DATA_ROOT, TABLE_ROOT, FIGURE_ROOT, OPTION_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

FINAL_MANIFEST_PATH = HOLDOUT_ROOT / "fresh_holdout_manifest.json"
if FINAL_MANIFEST_PATH.exists() and not ALLOW_OVERWRITE_FINALIZED_RUN:
    raise RuntimeError(
        "This date range already has a finalized holdout manifest. Preserve the first run. "
        "Only set ALLOW_OVERWRITE_FINALIZED_RUN=True to repair a documented technical error, "
        "never to change a disappointing result."
    )

load_dotenv(PROJECT_ROOT / "main.env")
API_KEY = os.getenv("MASSIVE_API_KEY", "").strip()
if not API_KEY:
    raise RuntimeError("MASSIVE_API_KEY was not found in main.env.")
client = MassiveREST(API_KEY)

print("Project root:", PROJECT_ROOT)
print("Fresh holdout:", HOLDOUT_START_DATE.date(), "to", HOLDOUT_END_DATE.date())
print("Output folder:", HOLDOUT_ROOT)


In [ ]:
# I verify the frozen artifacts before touching the new outcomes.
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


CLASSICAL_ROOT = PROJECT_ROOT / "models" / "classical"
CLASSICAL_MANIFEST_PATH = CLASSICAL_ROOT / "manifest.json"
classical_manifest = json.loads(CLASSICAL_MANIFEST_PATH.read_text(encoding="utf-8"))
classical_specs = {row["research_question"]: row for row in classical_manifest["artifacts"]}

for rq, spec in classical_specs.items():
    artifact_path = PROJECT_ROOT / spec["artifact_path"]
    assert artifact_path.exists(), f"Missing frozen artifact: {artifact_path}"
    assert sha256_file(artifact_path) == spec["artifact_sha256"], f"Hash mismatch: {rq}"
    assert pd.Timestamp(spec["training_end"]) <= LAST_TRAINING_DATE
    assert "Test" in spec["excluded_splits"]
    assert any("holdout" in item.lower() for item in spec["excluded_splits"])

protocol = pd.DataFrame(
    [
        {
            "research_question": rq,
            "model_family": spec["model_family"],
            "dataset_variant": spec["dataset_variant"],
            "training_end": spec["training_end"],
            "classification_threshold": spec["classification_threshold"],
            "artifact_hash_verified": True,
        }
        for rq, spec in classical_specs.items()
    ]
)
display(protocol)


## 2. Download only the post-freeze underlying data

I keep these files in a separate holdout folder. The downloader skips an existing ticker-date file unless `FORCE_REDOWNLOAD` is deliberately enabled. Empty market holidays and API errors remain visible in the audit.


In [ ]:
TICKERS = {
    "SPY": ("stock", "stock_minute_1"),
    "I:SPX": ("index", "index_minute_1"),
    "I:VIX": ("index", "index_minute_1"),
}


def safe_ticker_name(ticker: str) -> str:
    return ticker.replace(":", "_").replace("/", "_")


def raw_path(dataset: str, ticker: str, session_date: str) -> Path:
    return (
        HOLDOUT_DATA_ROOT
        / "raw"
        / dataset
        / f"ticker={safe_ticker_name(ticker)}"
        / f"date={session_date}"
        / "data.parquet"
    )


download_rows = []
requested_dates = pd.bdate_range(HOLDOUT_START_DATE, HOLDOUT_END_DATE)

if DOWNLOAD_UNDERLYINGS:
    for session_day in requested_dates:
        date_text = session_day.strftime("%Y-%m-%d")
        for ticker, (asset_class, dataset) in TICKERS.items():
            destination = raw_path(dataset, ticker, date_text)
            if destination.exists() and not FORCE_REDOWNLOAD:
                existing = pd.read_parquet(destination)
                download_rows.append(
                    {"session_date": date_text, "ticker": ticker, "status": "existing", "rows": len(existing)}
                )
                continue
            try:
                records = client.aggregates(ticker, date_text, date_text)
                frame = normalize_aggregates(records, ticker, asset_class)
                if frame.empty:
                    download_rows.append(
                        {"session_date": date_text, "ticker": ticker, "status": "empty", "rows": 0}
                    )
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                frame.to_parquet(destination, index=False)
                download_rows.append(
                    {"session_date": date_text, "ticker": ticker, "status": "downloaded", "rows": len(frame)}
                )
            except Exception as exc:
                download_rows.append(
                    {
                        "session_date": date_text,
                        "ticker": ticker,
                        "status": "error",
                        "rows": 0,
                        "message": str(exc)[:500],
                    }
                )

else:
    for session_day in requested_dates:
        date_text = session_day.strftime("%Y-%m-%d")
        for ticker, (_, dataset) in TICKERS.items():
            destination = raw_path(dataset, ticker, date_text)
            rows = len(pd.read_parquet(destination)) if destination.exists() else 0
            download_rows.append(
                {
                    "session_date": date_text,
                    "ticker": ticker,
                    "status": "existing" if destination.exists() else "missing",
                    "rows": rows,
                }
            )

download_audit = pd.DataFrame(download_rows)
download_audit.to_csv(TABLE_ROOT / "underlying_download_audit.csv", index=False)
display(download_audit.groupby(["ticker", "status"]).size().rename("requests").to_frame())

errors = download_audit[download_audit["status"].eq("error")]
if len(errors):
    display(errors)
    raise RuntimeError("One or more underlying requests failed. Fix the errors before scoring the holdout.")
missing = download_audit[download_audit["status"].eq("missing")]
if len(missing):
    display(missing)
    raise RuntimeError("Underlying downloading is disabled, but one or more required files are missing.")


## 3. Backward alignment and daily feature construction

SPY remains the canonical minute clock. SPX and VIX are matched only to observations at or before each SPY minute, with the same two-minute maximum staleness used in the dissertation pipeline. Features stop at the completed 14:59 bar and the outcome ends at 15:59.


In [ ]:
def read_holdout_ticker(dataset: str, ticker: str) -> pd.DataFrame:
    files = sorted((HOLDOUT_DATA_ROOT / "raw" / dataset / f"ticker={safe_ticker_name(ticker)}").rglob("data.parquet"))
    if not files:
        raise FileNotFoundError(f"No holdout files found for {ticker}")
    frame = pd.concat([pd.read_parquet(path) for path in files], ignore_index=True)
    frame["timestamp_utc"] = pd.to_datetime(frame["timestamp_utc"], utc=True)
    frame["timestamp_et"] = frame["timestamp_utc"].dt.tz_convert(TIMEZONE).dt.tz_localize(None)
    frame["session_date"] = frame["timestamp_et"].dt.strftime("%Y-%m-%d")
    frame = frame[
        frame["session_date"].between(
            HOLDOUT_START_DATE.strftime("%Y-%m-%d"),
            HOLDOUT_END_DATE.strftime("%Y-%m-%d"),
        )
    ].copy()
    minute = frame["timestamp_et"].dt.hour * 60 + frame["timestamp_et"].dt.minute
    return frame[minute.between(9 * 60 + 30, 15 * 60 + 59)].copy()


spy_raw = read_holdout_ticker("stock_minute_1", "SPY")
spx_raw = read_holdout_ticker("index_minute_1", "I:SPX")
vix_raw = read_holdout_ticker("index_minute_1", "I:VIX")

spy = spy_raw.rename(
    columns={
        "open": "spy_open", "high": "spy_high", "low": "spy_low", "close": "spy_close",
        "volume": "spy_volume", "vwap": "spy_vwap", "transactions": "spy_transactions",
    }
)[[
    "timestamp_utc", "timestamp_et", "session_date", "spy_open", "spy_high", "spy_low",
    "spy_close", "spy_volume", "spy_vwap", "spy_transactions",
]]
spx = spx_raw.rename(
    columns={
        "timestamp_utc": "spx_timestamp_utc", "open": "spx_open", "high": "spx_high",
        "low": "spx_low", "close": "spx_close",
    }
)[["spx_timestamp_utc", "session_date", "spx_open", "spx_high", "spx_low", "spx_close"]]
vix = vix_raw.rename(
    columns={
        "timestamp_utc": "vix_timestamp_utc", "open": "vix_open", "high": "vix_high",
        "low": "vix_low", "close": "vix_close",
    }
)[["vix_timestamp_utc", "session_date", "vix_open", "vix_high", "vix_low", "vix_close"]]

spy = spy.drop_duplicates("timestamp_utc", keep="last").sort_values("timestamp_utc")
spx = spx.drop_duplicates("spx_timestamp_utc", keep="last").sort_values("spx_timestamp_utc")
vix = vix.drop_duplicates("vix_timestamp_utc", keep="last").sort_values("vix_timestamp_utc")

aligned = pd.merge_asof(
    spy,
    spx,
    left_on="timestamp_utc",
    right_on="spx_timestamp_utc",
    by="session_date",
    direction="backward",
    tolerance=pd.Timedelta("2 minutes"),
)
aligned = pd.merge_asof(
    aligned.sort_values("timestamp_utc"),
    vix,
    left_on="timestamp_utc",
    right_on="vix_timestamp_utc",
    by="session_date",
    direction="backward",
    tolerance=pd.Timedelta("2 minutes"),
)
aligned["spx_staleness_seconds"] = (
    aligned["timestamp_utc"] - aligned["spx_timestamp_utc"]
).dt.total_seconds()
aligned["vix_staleness_seconds"] = (
    aligned["timestamp_utc"] - aligned["vix_timestamp_utc"]
).dt.total_seconds()
aligned["clock_minute"] = aligned["timestamp_et"].dt.hour * 60 + aligned["timestamp_et"].dt.minute

assert not (aligned["spx_timestamp_utc"] > aligned["timestamp_utc"]).fillna(False).any()
assert not (aligned["vix_timestamp_utc"] > aligned["timestamp_utc"]).fillna(False).any()
assert aligned["spx_staleness_seconds"].dropna().le(120).all()
assert aligned["vix_staleness_seconds"].dropna().le(120).all()
aligned.to_parquet(HOLDOUT_DATA_ROOT / "aligned_underlyings_1min.parquet", index=False)

display(
    pd.DataFrame(
        {
            "metric": ["aligned rows", "sessions", "missing SPX", "missing VIX"],
            "value": [
                len(aligned), aligned["session_date"].nunique(),
                aligned["spx_close"].isna().sum(), aligned["vix_close"].isna().sum(),
            ],
        }
    )
)


In [ ]:
def last_value_at_or_before(frame, column, timestamp):
    values = frame.loc[frame["timestamp_utc"].le(timestamp), ["timestamp_utc", column]].dropna(subset=[column])
    return np.nan if values.empty else values.iloc[-1][column]


def return_over_minutes(frame, column, end_timestamp, minutes):
    end_value = last_value_at_or_before(frame, column, end_timestamp)
    start_value = last_value_at_or_before(frame, column, end_timestamp - pd.Timedelta(minutes=minutes))
    if pd.isna(end_value) or pd.isna(start_value) or start_value == 0:
        return np.nan
    return end_value / start_value - 1


def realized_volatility(frame, column, end_timestamp, minutes):
    values = (
        frame.loc[
            frame["timestamp_utc"].between(end_timestamp - pd.Timedelta(minutes=minutes), end_timestamp),
            ["timestamp_utc", column],
        ]
        .dropna(subset=[column])
        .drop_duplicates("timestamp_utc")
        .sort_values("timestamp_utc")
    )
    if len(values) < 3:
        return np.nan
    return float(np.sqrt(np.square(np.log(values[column]).diff().dropna()).sum()))


def prior_window_return(frame, column, end_timestamp, minutes):
    recent_start = last_value_at_or_before(frame, column, end_timestamp - pd.Timedelta(minutes=minutes))
    prior_start = last_value_at_or_before(frame, column, end_timestamp - pd.Timedelta(minutes=2 * minutes))
    if pd.isna(recent_start) or pd.isna(prior_start) or prior_start == 0:
        return np.nan
    return recent_start / prior_start - 1


def create_daily_record(session_date, group):
    group = group.sort_values("timestamp_utc").copy()
    feature_window = group[group["clock_minute"] < 15 * 60]
    close_window = group[group["clock_minute"] < 16 * 60]
    if feature_window.empty or close_window.empty:
        return None
    if feature_window["clock_minute"].max() < 14 * 60 + 59 or close_window["clock_minute"].max() < 15 * 60 + 59:
        return None
    end_timestamp = feature_window["timestamp_utc"].max()
    spx_features = feature_window.dropna(subset=["spx_close"])
    vix_features = feature_window.dropna(subset=["vix_close"])
    spy_features = feature_window.dropna(subset=["spy_close"])
    spx_close_window = close_window.dropna(subset=["spx_close"])
    if any(frame.empty for frame in [spx_features, vix_features, spy_features, spx_close_window]):
        return None

    spx_at_1500 = float(spx_features.iloc[-1]["spx_close"])
    spx_at_close = float(spx_close_window.iloc[-1]["spx_close"])
    vix_at_1500 = float(vix_features.iloc[-1]["vix_close"])
    spy_at_1500 = float(spy_features.iloc[-1]["spy_close"])
    spx_open = float(spx_features["spx_open"].dropna().iloc[0])
    spy_open = float(spy_features["spy_open"].dropna().iloc[0])
    spx_high = float(spx_features["spx_high"].max())
    spx_low = float(spx_features["spx_low"].min())
    intraday_range = spx_high - spx_low

    spy_volume = pd.to_numeric(feature_window["spy_volume"], errors="coerce").fillna(0)
    spy_vwap = pd.to_numeric(feature_window["spy_vwap"], errors="coerce")
    valid_vwap = spy_vwap.notna() & spy_volume.gt(0)
    cumulative_vwap = (
        float(np.average(spy_vwap[valid_vwap], weights=spy_volume[valid_vwap]))
        if valid_vwap.any() else np.nan
    )
    last_30 = feature_window[feature_window["timestamp_utc"].gt(end_timestamp - pd.Timedelta(minutes=30))]
    last_60 = feature_window[feature_window["timestamp_utc"].gt(end_timestamp - pd.Timedelta(minutes=60))]
    earlier_volume = pd.to_numeric(
        feature_window.loc[
            feature_window["timestamp_utc"].le(end_timestamp - pd.Timedelta(minutes=60)), "spy_volume"
        ], errors="coerce"
    )
    expected_last_60 = earlier_volume.mean() * 60 if len(earlier_volume) else np.nan
    last_60_volume = pd.to_numeric(last_60["spy_volume"], errors="coerce").sum()
    volume_acceleration = (
        last_60_volume / expected_last_60 - 1
        if pd.notna(expected_last_60) and expected_last_60 > 0 else np.nan
    )
    previous_close = spx_features["spx_close"].shift(1)
    true_range = pd.concat(
        [
            spx_features["spx_high"] - spx_features["spx_low"],
            (spx_features["spx_high"] - previous_close).abs(),
            (spx_features["spx_low"] - previous_close).abs(),
        ], axis=1,
    ).max(axis=1)
    ret_last_60m = return_over_minutes(feature_window, "spx_close", end_timestamp, 60)
    ret_prior_60m = prior_window_return(feature_window, "spx_close", end_timestamp, 60)
    final_hour_return = spx_at_close / spx_at_1500 - 1

    return {
        "session_date": pd.Timestamp(session_date),
        "feature_cutoff_timestamp_utc": end_timestamp,
        "spx_at_1500": spx_at_1500,
        "spx_at_close": spx_at_close,
        "vix_level_1500": vix_at_1500,
        "spy_at_1500": spy_at_1500,
        "final_hour_return": final_hour_return,
        "target_up": int(final_hour_return > 0),
        "target_magnitude_bps": abs(final_hour_return) * 10_000,
        "target_large_move": int(abs(final_hour_return) * 10_000 >= LARGE_MOVE_THRESHOLD_BPS),
        "ret_open_to_1500": spx_at_1500 / spx_open - 1,
        "ret_last_15m": return_over_minutes(feature_window, "spx_close", end_timestamp, 15),
        "ret_last_30m": return_over_minutes(feature_window, "spx_close", end_timestamp, 30),
        "ret_last_60m": ret_last_60m,
        "vix_ret_last_15m": return_over_minutes(feature_window, "vix_close", end_timestamp, 15),
        "vix_ret_last_30m": return_over_minutes(feature_window, "vix_close", end_timestamp, 30),
        "vix_ret_last_60m": return_over_minutes(feature_window, "vix_close", end_timestamp, 60),
        "realized_vol_30m": realized_volatility(feature_window, "spx_close", end_timestamp, 30),
        "realized_vol_60m": realized_volatility(feature_window, "spx_close", end_timestamp, 60),
        "realized_vol_120m": realized_volatility(feature_window, "spx_close", end_timestamp, 120),
        "rv_open_to_1500": realized_volatility(
            feature_window, "spx_close", end_timestamp,
            int((end_timestamp - feature_window["timestamp_utc"].min()).total_seconds() // 60),
        ),
        "spy_ret_open_to_1500": spy_at_1500 / spy_open - 1,
        "spy_volume_last_30m": pd.to_numeric(last_30["spy_volume"], errors="coerce").sum(),
        "spy_volume_last_60m": last_60_volume,
        "spy_cum_volume_to_1500": spy_volume.sum(),
        "spy_volume_accel_60m_vs_avg": volume_acceleration,
        "spy_dist_from_vwap_pct": spy_at_1500 / cumulative_vwap - 1 if pd.notna(cumulative_vwap) else np.nan,
        "position_in_day_range": (spx_at_1500 - spx_low) / intraday_range if intraday_range > 0 else np.nan,
        "dist_from_day_high_pct": spx_at_1500 / spx_high - 1,
        "dist_from_day_low_pct": spx_at_1500 / spx_low - 1,
        "dist_from_open_pct": spx_at_1500 / spx_open - 1,
        "momentum_accel_60m_vs_prior": ret_last_60m - ret_prior_60m,
        "atr_open_to_1500": float(true_range.mean()),
        # This is the exact derived ATR feature used when the
        # saved classical pipelines were developed.
        "atr_pct_open_to_1500": float(true_range.mean()) / spx_at_1500,
    }


daily = pd.DataFrame(
    [
        record
        for session_date, group in aligned.groupby("session_date", sort=True)
        if (record := create_daily_record(session_date, group)) is not None
    ]
).sort_values("session_date").reset_index(drop=True)

if daily.empty:
    raise RuntimeError("No complete daily holdout observations were created.")
assert daily["session_date"].min() > LAST_PREVIOUSLY_VIEWED_DATE
reconstructed = daily["spx_at_close"] / daily["spx_at_1500"] - 1
assert np.allclose(daily["final_hour_return"], reconstructed)
display(daily[["session_date", "final_hour_return", "target_up", "target_magnitude_bps", "target_large_move"]])


## 4. Apply the frozen strict and relaxed quality rules

RQ1 and RQ2 use the relaxed variant, while RQ3 uses the strict variant. Both variants still require 390 SPY minutes and exact SPX observations at 14:59 and 15:59. The relaxed rule only permits up to five missing VIX minutes in the final two hours when the four required VIX anchors remain timely.


In [ ]:
FULL_SESSION_ROWS = 390
MAX_RELAXED_VIX_GAPS = 5
VIX_ANCHORS = [13 * 60 + 59, 14 * 60 + 29, 14 * 60 + 44, 14 * 60 + 59]


def exact_spx_anchor(group, minute):
    anchor = group[group["clock_minute"].eq(minute)]
    return bool(
        len(anchor)
        and anchor["spx_close"].notna().any()
        and anchor["spx_timestamp_utc"].eq(anchor["timestamp_utc"]).any()
    )


def valid_vix_anchor(group, minute):
    anchor = group[
        group["clock_minute"].eq(minute)
        & group["vix_close"].notna()
        & group["vix_timestamp_utc"].notna()
    ]
    if anchor.empty:
        return False
    staleness = (anchor["timestamp_utc"] - anchor["vix_timestamp_utc"]).dt.total_seconds()
    return bool(staleness.between(0, 120).all())


quality_rows = []
for session_date, group in aligned.groupby("session_date", sort=True):
    final_120 = group[group["clock_minute"].between(13 * 60, 14 * 60 + 59)]
    row = {
        "session_date": pd.Timestamp(session_date),
        "rows": len(group),
        "exact_spx_1459": exact_spx_anchor(group, 14 * 60 + 59),
        "exact_spx_1559": exact_spx_anchor(group, 15 * 60 + 59),
        "missing_spx_final_120m": int(final_120["spx_close"].isna().sum()),
        "missing_vix_final_120m": int(final_120["vix_close"].isna().sum()),
    }
    row["vix_anchors_valid"] = all(valid_vix_anchor(group, minute) for minute in VIX_ANCHORS)
    row["strict_eligible"] = (
        row["rows"] == FULL_SESSION_ROWS
        and row["exact_spx_1459"] and row["exact_spx_1559"]
        and row["missing_spx_final_120m"] == 0
        and row["missing_vix_final_120m"] == 0
    )
    row["relaxed_eligible"] = (
        row["rows"] == FULL_SESSION_ROWS
        and row["exact_spx_1459"] and row["exact_spx_1559"]
        and row["missing_spx_final_120m"] == 0
        and row["missing_vix_final_120m"] <= MAX_RELAXED_VIX_GAPS
        and row["vix_anchors_valid"]
    )
    quality_rows.append(row)

session_quality = pd.DataFrame(quality_rows)
assert not (session_quality["strict_eligible"] & ~session_quality["relaxed_eligible"]).any()
daily = daily.merge(session_quality, on="session_date", how="left", validate="one_to_one")
strict_daily = daily[daily["strict_eligible"]].copy()
relaxed_daily = daily[daily["relaxed_eligible"]].copy()
if relaxed_daily.empty or strict_daily.empty:
    raise RuntimeError("The holdout has no eligible strict or relaxed sessions.")

daily.to_parquet(HOLDOUT_DATA_ROOT / "daily_holdout_all_quality.parquet", index=False)
strict_daily.to_parquet(HOLDOUT_DATA_ROOT / "daily_holdout_strict.parquet", index=False)
relaxed_daily.to_parquet(HOLDOUT_DATA_ROOT / "daily_holdout_relaxed.parquet", index=False)
session_quality.to_csv(TABLE_ROOT / "session_quality.csv", index=False)
display(
    pd.Series(
        {
            "daily rows with targets": len(daily),
            "strict eligible": len(strict_daily),
            "relaxed eligible": len(relaxed_daily),
        }, name="sessions"
    ).to_frame()
)


## 5. Load the frozen classical models and score RQ1-RQ3

No `.fit()` call is permitted below. Feature order, model family and classification thresholds come from the saved manifest. RQ1 confidence is the absolute distance between its probability and its frozen classification threshold.


In [ ]:
models = {
    rq: joblib.load(PROJECT_ROOT / spec["artifact_path"])
    for rq, spec in classical_specs.items()
}


def positive_score(model, X):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X))[:, 1]
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X)).reshape(-1)
    raise TypeError("The classifier exposes neither predict_proba nor decision_function.")


rq1_spec = classical_specs["RQ1"]
rq2_spec = classical_specs["RQ2"]
rq3_spec = classical_specs["RQ3"]

rq1 = relaxed_daily.copy()
rq1["rq1_score"] = positive_score(models["RQ1"], rq1[rq1_spec["features"]])
rq1["rq1_prediction"] = (rq1["rq1_score"] >= float(rq1_spec["classification_threshold"])).astype(int)
rq1["rq1_confidence_raw"] = (rq1["rq1_score"] - float(rq1_spec["classification_threshold"])).abs()
rq1["rq1_high_confidence"] = rq1["rq1_confidence_raw"] >= RQ1_HIGH_CONFIDENCE_DISTANCE

rq2 = relaxed_daily.copy()
# Keep the saved regressor's raw prediction unchanged so this is
# directly comparable with its original evaluation and RQ4 rule.
rq2["rq2_predicted_move_bps"] = np.asarray(
    models["RQ2"].predict(rq2[rq2_spec["features"]])
).reshape(-1)

rq3 = strict_daily.copy()
rq3["rq3_score"] = positive_score(models["RQ3"], rq3[rq3_spec["features"]])
rq3["rq3_large_move_prediction"] = (
    rq3["rq3_score"] >= float(rq3_spec["classification_threshold"])
).astype(int)

prediction_columns = [
    "session_date", "final_hour_return", "target_up", "target_magnitude_bps", "target_large_move",
    "spx_at_1500", "spx_at_close", "ret_last_60m",
]
holdout_predictions = (
    rq1[prediction_columns + ["rq1_score", "rq1_prediction", "rq1_confidence_raw", "rq1_high_confidence"]]
    .merge(
        rq2[["session_date", "rq2_predicted_move_bps"]],
        on="session_date", how="inner", validate="one_to_one",
    )
    .merge(
        rq3[["session_date", "rq3_score", "rq3_large_move_prediction"]],
        on="session_date", how="inner", validate="one_to_one",
    )
    .sort_values("session_date")
    .reset_index(drop=True)
)
holdout_predictions["mean_reversion_60m_prediction"] = (
    holdout_predictions["ret_last_60m"] < 0
).astype(int)
holdout_predictions.to_csv(TABLE_ROOT / "fresh_holdout_classical_predictions.csv", index=False)
display(holdout_predictions)


In [ ]:
def safe_balanced_accuracy(y_true, y_pred):
    return balanced_accuracy_score(y_true, y_pred) if pd.Series(y_true).nunique() > 1 else np.nan


def safe_average_precision(y_true, score):
    return average_precision_score(y_true, score) if pd.Series(y_true).nunique() > 1 else np.nan


def classification_summary(rq_name, y_true, prediction, score=None):
    return {
        "research_question": rq_name,
        "sessions": len(y_true),
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": safe_balanced_accuracy(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "average_precision": safe_average_precision(y_true, score) if score is not None else np.nan,
        "positive_prevalence": float(np.mean(y_true)),
    }


rq1_metrics = classification_summary(
    "RQ1 Direction",
    rq1["target_up"], rq1["rq1_prediction"], rq1["rq1_score"],
)
rq1_benchmark = classification_summary(
    "RQ1 60-minute mean reversion benchmark",
    rq1["target_up"], (rq1["ret_last_60m"] < 0).astype(int), None,
)

rq2_error = rq2["target_magnitude_bps"] - rq2["rq2_predicted_move_bps"]
spearman_value = spearmanr(
    rq2["target_magnitude_bps"], rq2["rq2_predicted_move_bps"], nan_policy="omit"
).statistic if len(rq2) > 1 else np.nan
rq2_metrics = {
    "research_question": "RQ2 Magnitude",
    "sessions": len(rq2),
    "mae_bps": mean_absolute_error(rq2["target_magnitude_bps"], rq2["rq2_predicted_move_bps"]),
    "rmse_bps": math.sqrt(mean_squared_error(rq2["target_magnitude_bps"], rq2["rq2_predicted_move_bps"])),
    "r2": r2_score(rq2["target_magnitude_bps"], rq2["rq2_predicted_move_bps"]) if len(rq2) > 1 else np.nan,
    "spearman": spearman_value,
    "mean_error_bps": float(rq2_error.mean()),
}

rq3_metrics = classification_summary(
    "RQ3 Large movement",
    rq3["target_large_move"], rq3["rq3_large_move_prediction"], rq3["rq3_score"],
)
rq3_metrics["average_precision_lift_over_prevalence"] = (
    rq3_metrics["average_precision"] - rq3_metrics["positive_prevalence"]
    if pd.notna(rq3_metrics["average_precision"]) else np.nan
)

classification_metrics = pd.DataFrame([rq1_metrics, rq1_benchmark, rq3_metrics])
regression_metrics = pd.DataFrame([rq2_metrics])
classification_metrics.to_csv(TABLE_ROOT / "fresh_holdout_classification_metrics.csv", index=False)
regression_metrics.to_csv(TABLE_ROOT / "fresh_holdout_regression_metrics.csv", index=False)
display(classification_metrics)
display(regression_metrics)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, y_true, y_pred in [
    (axes[0], "RQ1 direction", rq1["target_up"], rq1["rq1_prediction"]),
    (axes[1], "RQ3 large move", rq3["target_large_move"], rq3["rq3_large_move_prediction"]),
]:
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    image = ax.imshow(matrix, cmap="Blues")
    for row in range(2):
        for column in range(2):
            ax.text(column, row, matrix[row, column], ha="center", va="center")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "fresh_holdout_confusion_matrices.png", dpi=220, bbox_inches="tight")
plt.show()


## 6. Optional exploratory LSTM comparison

This section uses the same 13:00-14:59 sequence definition and the saved LSTM scalers. It does not train the networks. These results remain secondary because the LSTM artifacts were development refits rather than independently evaluated final models.


In [ ]:
lstm_metrics = pd.DataFrame()
lstm_predictions = pd.DataFrame()

if RUN_EXPLORATORY_LSTM:
    try:
        import tensorflow as tf
        from tensorflow import keras

        LSTM_ROOT = PROJECT_ROOT / "models" / "lstm"
        lstm_manifest = json.loads((LSTM_ROOT / "manifest.json").read_text(encoding="utf-8"))
        lstm_specs = {row["research_question"]: row for row in lstm_manifest["artifacts"]}
        sequence_features = lstm_specs["RQ1"]["features"]

        def build_session_minute_features(session_frame):
            frame = session_frame.sort_values("timestamp_utc").copy()
            frame = frame[frame["clock_minute"] <= 14 * 60 + 59].copy()
            numeric = [
                "spx_open", "spx_high", "spx_low", "spx_close", "spy_open", "spy_close",
                "spy_volume", "vix_close", "spx_staleness_seconds", "vix_staleness_seconds",
            ]
            frame[numeric] = frame[numeric].apply(pd.to_numeric, errors="coerce")
            frame["spx_logret_1m"] = np.log(frame["spx_close"]).diff()
            frame["spx_logret_5m"] = np.log(frame["spx_close"]).diff(5)
            frame["spy_logret_1m"] = np.log(frame["spy_close"]).diff()
            frame["vix_logret_1m"] = np.log(frame["vix_close"]).diff()
            frame["vix_logret_5m"] = np.log(frame["vix_close"]).diff(5)
            frame["vix_level"] = frame["vix_close"]
            frame["spx_from_open_pct"] = frame["spx_close"] / frame["spx_open"].dropna().iloc[0] - 1
            frame["spy_from_open_pct"] = frame["spy_close"] / frame["spy_open"].dropna().iloc[0] - 1
            running_high = frame["spx_high"].cummax()
            running_low = frame["spx_low"].cummin()
            running_range = running_high - running_low
            frame["spx_running_range_position"] = np.where(
                running_range > 0, (frame["spx_close"] - running_low) / running_range, 0.5
            )
            squared = frame["spx_logret_1m"] ** 2
            frame["spx_rv_15m"] = np.sqrt(squared.rolling(15, min_periods=10).sum())
            frame["spx_rv_30m"] = np.sqrt(squared.rolling(30, min_periods=20).sum())
            volume = frame["spy_volume"].clip(lower=0)
            frame["spy_log_volume"] = np.log1p(volume)
            frame["spy_log_cum_volume"] = np.log1p(volume.fillna(0).cumsum())
            sequence = frame[frame["clock_minute"].between(13 * 60, 14 * 60 + 59)].copy()
            if len(sequence):
                sequence["sequence_time_fraction"] = np.linspace(0.0, 1.0, len(sequence))
            return sequence

        sequence_rows = []
        sequence_dates = []
        strict_dates = set(strict_daily["session_date"].dt.strftime("%Y-%m-%d"))
        for session_date, group in aligned.groupby("session_date", sort=True):
            if session_date not in strict_dates:
                continue
            sequence = build_session_minute_features(group)
            values = sequence[sequence_features].replace([np.inf, -np.inf], np.nan)
            if len(values) == 120 and not values.isna().any().any():
                sequence_rows.append(values.to_numpy(dtype=np.float32))
                sequence_dates.append(pd.Timestamp(session_date))

        if sequence_rows:
            X_sequence = np.stack(sequence_rows).astype(np.float32)
            sequence_truth = strict_daily.set_index("session_date").loc[sequence_dates].reset_index()
            prediction_frame = sequence_truth[
                ["session_date", "target_up", "target_magnitude_bps", "target_large_move"]
            ].copy()
            metric_rows = []
            for rq in ["RQ1", "RQ2", "RQ3"]:
                spec = lstm_specs[rq]
                model_path = PROJECT_ROOT / spec["artifact_path"]
                scaler_path = PROJECT_ROOT / spec["scaler_path"]
                assert sha256_file(model_path) == spec["artifact_sha256"]
                assert sha256_file(scaler_path) == spec["scaler_sha256"]
                scaler = joblib.load(scaler_path)
                X_scaled = scaler.transform(X_sequence.reshape(-1, X_sequence.shape[-1])).reshape(X_sequence.shape)
                model = keras.models.load_model(model_path)
                score = model.predict(X_scaled, verbose=0).reshape(-1)
                prediction_frame[f"{rq.lower()}_score"] = score
                if rq == "RQ1":
                    pred = (score >= float(spec["classification_threshold"])).astype(int)
                    prediction_frame["rq1_prediction"] = pred
                    metric_rows.append(classification_summary("LSTM RQ1", sequence_truth["target_up"], pred, score))
                elif rq == "RQ2":
                    prediction_frame["rq2_predicted_move_bps"] = score
                    metric_rows.append(
                        {
                            "research_question": "LSTM RQ2", "sessions": len(score),
                            "mae_bps": mean_absolute_error(sequence_truth["target_magnitude_bps"], score),
                            "rmse_bps": math.sqrt(mean_squared_error(sequence_truth["target_magnitude_bps"], score)),
                        }
                    )
                else:
                    pred = (score >= float(spec["classification_threshold"])).astype(int)
                    prediction_frame["rq3_prediction"] = pred
                    metric_rows.append(classification_summary("LSTM RQ3", sequence_truth["target_large_move"], pred, score))
            lstm_predictions = prediction_frame
            lstm_metrics = pd.DataFrame(metric_rows)
            lstm_predictions.to_csv(TABLE_ROOT / "fresh_holdout_lstm_predictions.csv", index=False)
            lstm_metrics.to_csv(TABLE_ROOT / "fresh_holdout_lstm_metrics.csv", index=False)
            display(lstm_metrics)
        else:
            print("No complete strict 120-minute sequences were available.")
    except (ImportError, ModuleNotFoundError) as exc:
        print("TensorFlow is unavailable, so the optional LSTM comparison was skipped:", exc)
else:
    print("Optional LSTM evaluation is disabled.")


## 7. RQ4: economic usefulness on the untouched dates

The main combined signal is frozen: RQ1 must exceed the development confidence-distance cutoff, RQ2 must predict at least 20 basis points, and RQ3 must flag a large move. I report coverage, realised opportunity precision, lift and direction accuracy without changing the rule.


In [ ]:
holdout_predictions["combined_signal"] = (
    holdout_predictions["rq1_high_confidence"]
    & holdout_predictions["rq2_predicted_move_bps"].ge(RQ2_MIN_PREDICTED_MOVE_BPS)
    & holdout_predictions["rq3_large_move_prediction"].eq(1)
)
holdout_predictions["actual_ge_20bps"] = holdout_predictions["target_magnitude_bps"].ge(20)
holdout_predictions["actual_ge_30bps"] = holdout_predictions["target_magnitude_bps"].ge(30)
selected = holdout_predictions[holdout_predictions["combined_signal"]].copy()


def precision_and_lift(flag_column):
    base = float(holdout_predictions[flag_column].mean())
    precision = float(selected[flag_column].mean()) if len(selected) else np.nan
    lift = precision / base if base > 0 and pd.notna(precision) else np.nan
    return base, precision, lift


base20, precision20, lift20 = precision_and_lift("actual_ge_20bps")
base30, precision30, lift30 = precision_and_lift("actual_ge_30bps")
rq4_summary = pd.DataFrame(
    [
        {
            "eligible_sessions": len(holdout_predictions),
            "selected_sessions": len(selected),
            "coverage": len(selected) / len(holdout_predictions),
            "direction_accuracy_selected": (
                accuracy_score(selected["target_up"], selected["rq1_prediction"]) if len(selected) else np.nan
            ),
            "mean_realised_move_selected_bps": selected["target_magnitude_bps"].mean(),
            "base_rate_ge_20bps": base20,
            "selected_precision_ge_20bps": precision20,
            "lift_ge_20bps": lift20,
            "base_rate_ge_30bps": base30,
            "selected_precision_ge_30bps": precision30,
            "lift_ge_30bps": lift30,
        }
    ]
)
rq4_summary.to_csv(TABLE_ROOT / "fresh_holdout_rq4_summary.csv", index=False)
selected.to_csv(TABLE_ROOT / "fresh_holdout_rq5_candidate_sessions.csv", index=False)
display(rq4_summary)
display(selected)


## 8. RQ5: retrieve candidate options and test the frozen strategies

Only sessions selected by the frozen combined rule reach this stage. I retrieve calls and puts so the RQ1 direction can be compared with the fixed 60-minute mean-reversion direction on the same dates. ATM is primary; 5-30 point OTM contracts are sensitivities. Trade-derived minute aggregates are reference prices, so the original synthetic cost scenarios remain necessary.


In [ ]:
EXECUTION_SCENARIOS = {
    "frictionless": {"min_penalty_points": 0.0, "pct_of_reference_price": 0.0, "commission_per_side_usd": 0.0},
    "low_cost": {"min_penalty_points": 0.05, "pct_of_reference_price": 0.02, "commission_per_side_usd": 1.0},
    "medium_cost": {"min_penalty_points": 0.10, "pct_of_reference_price": 0.05, "commission_per_side_usd": 1.5},
    "severe_cost": {"min_penalty_points": 0.20, "pct_of_reference_price": 0.10, "commission_per_side_usd": 2.0},
}
REFERENCE_UNDERLYING_CANDIDATES = ["SPX", "SPXW"]


def fetch_same_day_contracts(session_date):
    date_text = pd.Timestamp(session_date).strftime("%Y-%m-%d")
    attempts = []
    # Keep the same query priority used in the original RQ5 retrieval:
    # both point-in-time candidates first, then the expired fallback.
    for mode in ["as_of", "expired"]:
        for underlying in REFERENCE_UNDERLYING_CANDIDATES:
            params = {
                "underlying_ticker": underlying,
                "expiration_date": date_text,
                "limit": 1000,
                "sort": "strike_price",
                "order": "asc",
            }
            params["as_of" if mode == "as_of" else "expired"] = date_text if mode == "as_of" else "true"
            try:
                rows = list(client.paginate("/v3/reference/options/contracts", params=params))
                attempts.append({"underlying": underlying, "mode": mode, "status": "ok", "rows": len(rows)})
            except Exception as exc:
                rows = []
                attempts.append(
                    {"underlying": underlying, "mode": mode, "status": "error", "message": str(exc)[:300]}
                )
            if rows:
                frame = pd.DataFrame(rows)
                frame["reference_query_underlying"] = underlying
                frame["reference_query_mode"] = mode
                return frame, attempts
    return pd.DataFrame(), attempts


def select_contract(contracts, option_type, spot, offset):
    side = contracts[contracts["contract_type"].astype(str).str.lower().eq(option_type)].copy()
    side["strike_price"] = pd.to_numeric(side["strike_price"], errors="coerce")
    side = side.dropna(subset=["strike_price"])
    if side.empty:
        return None
    target = spot + offset if option_type == "call" else spot - offset
    side["distance"] = (side["strike_price"] - target).abs()
    chosen = side.sort_values(["distance", "strike_price", "ticker"]).iloc[0].to_dict()
    chosen.update({"target_strike": target, "spot_for_selection": spot, "otm_offset_points": offset})
    return chosen


option_selection_rows = []
option_bar_frames = []

if DOWNLOAD_OPTIONS_FOR_RQ5 and len(selected):
    for candidate in selected.itertuples(index=False):
        contracts, query_attempts = fetch_same_day_contracts(candidate.session_date)
        if contracts.empty:
            option_selection_rows.append(
                {
                    "session_date": candidate.session_date,
                    "selection_status": "no_contracts",
                    "query_attempts": json.dumps(query_attempts),
                }
            )
            continue
        for option_type in ["call", "put"]:
            for offset in OPTION_OFFSETS:
                chosen = select_contract(contracts, option_type, candidate.spx_at_1500, offset)
                if chosen is None:
                    continue
                ticker = chosen["ticker"]
                date_text = pd.Timestamp(candidate.session_date).strftime("%Y-%m-%d")
                try:
                    records = client.aggregates(ticker, date_text, date_text)
                except Exception as exc:
                    option_selection_rows.append(
                        {
                            "session_date": candidate.session_date,
                            "selection_status": "bar_download_error",
                            "ticker": ticker,
                            "option_type": option_type,
                            "otm_offset_points": offset,
                            "message": str(exc)[:300],
                        }
                    )
                    continue
                bars = normalize_aggregates(records, ticker, "option")
                if len(bars):
                    bars["timestamp"] = pd.to_datetime(bars["timestamp_utc"], utc=True).dt.tz_convert(TIMEZONE)
                    hhmm = bars["timestamp"].dt.strftime("%H:%M")
                    bars = bars[hhmm.between("14:55", "16:00")].copy()
                    bars["option_ticker"] = ticker
                    bars["option_type"] = option_type
                    bars["otm_offset_points"] = offset
                    bars["session_date"] = pd.Timestamp(candidate.session_date)
                    option_bar_frames.append(bars)
                option_selection_rows.append(
                    {
                        "session_date": candidate.session_date,
                        "selection_status": "selected",
                        "ticker": ticker,
                        "option_type": option_type,
                        "otm_offset_points": offset,
                        "strike_price": chosen["strike_price"],
                        "spot_for_selection": candidate.spx_at_1500,
                        "reference_query_underlying": chosen.get("reference_query_underlying"),
                        "reference_query_mode": chosen.get("reference_query_mode"),
                        "bars": len(bars),
                    }
                )

option_selection = pd.DataFrame(option_selection_rows)
option_bars = pd.concat(option_bar_frames, ignore_index=True) if option_bar_frames else pd.DataFrame()
if len(option_bars):
    option_bars = option_bars.drop_duplicates(
        ["session_date", "option_ticker", "timestamp_utc"]
    ).reset_index(drop=True)
option_selection.to_csv(OPTION_ROOT / "fresh_holdout_contract_selection.csv", index=False)
if len(option_bars):
    option_bars.to_parquet(OPTION_ROOT / "fresh_holdout_option_minute_bars.parquet", index=False)
display(option_selection)


In [ ]:
def choose_reference_bars(frame):
    frame = frame.sort_values("timestamp").copy()
    hhmm = frame["timestamp"].dt.strftime("%H:%M")
    entry = frame[hhmm.between("15:00", "15:05")].head(1)
    exit_ = frame[hhmm.between("15:55", "15:59")].tail(1)
    if entry.empty or exit_.empty:
        return None
    entry_price = float(entry.iloc[0]["open"])
    exit_price = float(exit_.iloc[0]["close"])
    if not np.isfinite(entry_price) or not np.isfinite(exit_price) or entry_price <= 0 or exit_price < 0:
        return None
    return {
        "entry_timestamp": entry.iloc[0]["timestamp"],
        "exit_timestamp": exit_.iloc[0]["timestamp"],
        "entry_reference_price": entry_price,
        "exit_reference_price": exit_price,
    }


def executed_trade(entry_reference, exit_reference, scenario):
    params = EXECUTION_SCENARIOS[scenario]
    entry_penalty = max(params["min_penalty_points"], params["pct_of_reference_price"] * entry_reference)
    exit_penalty = max(params["min_penalty_points"], params["pct_of_reference_price"] * exit_reference)
    entry_execution = entry_reference + entry_penalty
    exit_execution = max(0.0, exit_reference - exit_penalty)
    commission = 2 * params["commission_per_side_usd"]
    capital = entry_execution * CONTRACT_MULTIPLIER + params["commission_per_side_usd"]
    pnl = (exit_execution - entry_execution) * CONTRACT_MULTIPLIER - commission
    return entry_execution, exit_execution, capital, pnl, pnl / capital


trade_rows = []
if len(option_bars):
    option_bars["timestamp"] = pd.to_datetime(option_bars["timestamp"])
    selected_indexed = selected.set_index("session_date")
    for selection_row in option_selection[option_selection["selection_status"].eq("selected")].itertuples(index=False):
        date = pd.Timestamp(selection_row.session_date)
        candidate = selected_indexed.loc[date]
        contract_bars = option_bars[
            option_bars["session_date"].eq(date)
            & option_bars["option_ticker"].eq(selection_row.ticker)
        ].copy()
        reference = choose_reference_bars(contract_bars)
        if reference is None:
            continue
        strategies = {
            "RQ1_ML": "call" if int(candidate.rq1_prediction) == 1 else "put",
            "MeanReversion60m": "call" if int(candidate.mean_reversion_60m_prediction) == 1 else "put",
        }
        for strategy, required_type in strategies.items():
            if selection_row.option_type != required_type:
                continue
            for scenario in EXECUTION_SCENARIOS:
                entry_execution, exit_execution, capital, pnl, return_on_premium = executed_trade(
                    reference["entry_reference_price"], reference["exit_reference_price"], scenario
                )
                trade_rows.append(
                    {
                        "session_date": date,
                        "strategy": strategy,
                        "option_type": selection_row.option_type,
                        "option_ticker": selection_row.ticker,
                        "strike_price": selection_row.strike_price,
                        "otm_offset_points": selection_row.otm_offset_points,
                        "execution_scenario": scenario,
                        **reference,
                        "entry_execution_price": entry_execution,
                        "exit_execution_price": exit_execution,
                        "capital_at_risk_usd": capital,
                        "net_pnl_usd": pnl,
                        "net_return_on_premium": return_on_premium,
                    }
                )

option_trades = pd.DataFrame(trade_rows)


def summarize_trades(frame):
    ordered = frame.sort_values("session_date")
    equity = ordered["net_pnl_usd"].cumsum()
    drawdown = equity - equity.cummax()
    losses = -frame.loc[frame["net_pnl_usd"] < 0, "net_pnl_usd"].sum()
    gains = frame.loc[frame["net_pnl_usd"] > 0, "net_pnl_usd"].sum()
    return {
        "trades": len(frame),
        "win_rate": float((frame["net_pnl_usd"] > 0).mean()),
        "total_pnl_usd": float(frame["net_pnl_usd"].sum()),
        "mean_pnl_usd": float(frame["net_pnl_usd"].mean()),
        "median_pnl_usd": float(frame["net_pnl_usd"].median()),
        "mean_return_on_premium": float(frame["net_return_on_premium"].mean()),
        "median_capital_at_risk_usd": float(frame["capital_at_risk_usd"].median()),
        "profit_factor": float(gains / losses) if losses > 0 else np.inf,
        "max_drawdown_usd": float(drawdown.min()),
    }


if len(option_trades):
    option_summary_rows = []
    for keys, group in option_trades.groupby(["strategy", "otm_offset_points", "execution_scenario"]):
        strategy, offset, scenario = keys
        option_summary_rows.append(
            {"strategy": strategy, "otm_offset_points": offset, "execution_scenario": scenario, **summarize_trades(group)}
        )
    option_summary = pd.DataFrame(option_summary_rows)
    option_trades.to_csv(OPTION_ROOT / "fresh_holdout_option_trade_log.csv", index=False)
    option_summary.to_csv(OPTION_ROOT / "fresh_holdout_option_strategy_summary.csv", index=False)
    display(option_summary[
        option_summary["execution_scenario"].eq("medium_cost")
        & option_summary["otm_offset_points"].isin([0, 15, 30])
    ])

    primary = option_trades[
        option_trades["strategy"].eq("RQ1_ML")
        & option_trades["otm_offset_points"].eq(0)
        & option_trades["execution_scenario"].eq("medium_cost")
    ].sort_values("session_date")
    if len(primary):
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(primary["session_date"], primary["net_pnl_usd"].cumsum(), marker="o")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title("Fresh-holdout RQ5 cumulative P&L: ATM ML, medium costs")
        ax.set_xlabel("Session")
        ax.set_ylabel("Cumulative net P&L (USD)")
        fig.autofmt_xdate()
        fig.tight_layout()
        fig.savefig(FIGURE_ROOT / "fresh_holdout_rq5_cumulative_pnl.png", dpi=220, bbox_inches="tight")
        plt.show()
else:
    option_summary = pd.DataFrame()
    print("No usable fresh-holdout option trades were available. This is a valid RQ5 outcome and is not a reason to relax the rule.")


### Pre-specified stop/target sensitivity

The next cell compares the ATM medium-cost hold with the already-planned -50% stop and +100% target. If both are touched inside the same minute bar, the stop is assumed first. This is deliberately conservative and is not retuned from the fresh results.


In [ ]:
stop_target_rows = []
if len(option_trades):
    primary_hold = option_trades[
        option_trades["strategy"].eq("RQ1_ML")
        & option_trades["otm_offset_points"].eq(0)
        & option_trades["execution_scenario"].eq("medium_cost")
    ].copy()
    params = EXECUTION_SCENARIOS["medium_cost"]
    for trade in primary_hold.itertuples(index=False):
        path = option_bars[
            option_bars["session_date"].eq(pd.Timestamp(trade.session_date))
            & option_bars["option_ticker"].eq(trade.option_ticker)
        ].sort_values("timestamp").copy()
        path = path[
            path["timestamp"].gt(pd.Timestamp(trade.entry_timestamp))
            & path["timestamp"].le(pd.Timestamp(trade.exit_timestamp))
        ]
        stop_price = trade.entry_execution_price * 0.50
        target_price = trade.entry_execution_price * 2.00
        exit_reference = trade.exit_reference_price
        exit_reason = "close"
        exit_timestamp = trade.exit_timestamp
        for bar in path.itertuples(index=False):
            touched_stop = float(bar.low) <= stop_price
            touched_target = float(bar.high) >= target_price
            if touched_stop:
                exit_reference = min(stop_price, float(bar.open))
                exit_reason = "stop_-50%"
                exit_timestamp = bar.timestamp
                break
            if touched_target:
                exit_reference = target_price
                exit_reason = "target_+100%"
                exit_timestamp = bar.timestamp
                break
        _, exit_execution, capital, pnl, return_on_premium = executed_trade(
            trade.entry_reference_price, exit_reference, "medium_cost"
        )
        stop_target_rows.append(
            {
                "session_date": trade.session_date,
                "policy": "stop50_tp100",
                "exit_reason": exit_reason,
                "exit_timestamp": exit_timestamp,
                "capital_at_risk_usd": capital,
                "net_pnl_usd": pnl,
                "net_return_on_premium": return_on_premium,
            }
        )

stop_target_trades = pd.DataFrame(stop_target_rows)
if len(stop_target_trades):
    stop_target_summary = pd.DataFrame(
        [
            {"policy": "hold_to_close", **summarize_trades(primary_hold)},
            {"policy": "stop50_tp100", **summarize_trades(stop_target_trades)},
        ]
    )
    stop_target_trades.to_csv(OPTION_ROOT / "fresh_holdout_stop50_tp100_trade_log.csv", index=False)
    stop_target_summary.to_csv(OPTION_ROOT / "fresh_holdout_exit_policy_summary.csv", index=False)
    display(stop_target_summary)


## 9. Save the frozen-protocol manifest and interpretation tables

The manifest records exactly which dates, artifacts and rules produced this run. Small samples may make some metrics undefined or unstable. A lack of selected trades is evidence about coverage, not permission to loosen the filters after seeing the holdout.


In [ ]:
manifest = {
    "analysis": "Fresh post-17-Jul-2026 holdout evaluation",
    "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "holdout_start": HOLDOUT_START_DATE.strftime("%Y-%m-%d"),
    "holdout_end": HOLDOUT_END_DATE.strftime("%Y-%m-%d"),
    "holdout_consumed": True,
    "models_refitted": False,
    "classical_manifest_sha256": sha256_file(CLASSICAL_MANIFEST_PATH),
    "classical_artifact_hashes": {
        rq: spec["artifact_sha256"] for rq, spec in classical_specs.items()
    },
    "frozen_rules": {
        "rq1_high_confidence_distance": RQ1_HIGH_CONFIDENCE_DISTANCE,
        "rq2_min_predicted_move_bps": RQ2_MIN_PREDICTED_MOVE_BPS,
        "rq3_required": RQ3_REQUIRED,
        "large_move_threshold_bps": LARGE_MOVE_THRESHOLD_BPS,
        "rq5_primary_strike_offset_points": 0,
        "rq5_primary_execution_scenario": "medium_cost",
        "rq5_entry_window_et": ["15:00", "15:05"],
        "rq5_exit_window_et": ["15:55", "15:59"],
        "rq5_primary_alternative_exit": "stop50_tp100",
    },
    "row_counts": {
        "aligned_minutes": len(aligned),
        "daily_with_targets": len(daily),
        "strict_eligible": len(strict_daily),
        "relaxed_eligible": len(relaxed_daily),
        "rq4_intersection": len(holdout_predictions),
        "rq5_candidates": len(selected),
        "usable_primary_option_trades": int(len(primary_hold)) if "primary_hold" in globals() else 0,
    },
    "option_data_note": "Trade-derived minute aggregates; no historical NBBO quotes.",
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
FINAL_MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, default=str) + "\n", encoding="utf-8"
)

interpretation = pd.DataFrame(
    [
        ["RQ1", "Balanced accuracy, accuracy and comparison with 60-minute mean reversion", len(rq1)],
        ["RQ2", "MAE/RMSE in basis points, R-squared and Spearman association", len(rq2)],
        ["RQ3", "Average precision relative to large-move prevalence, plus recall and F1", len(rq3)],
        ["RQ4", "Frozen combined-rule coverage, opportunity precision and lift", len(holdout_predictions)],
        ["RQ5", "Net P&L, win rate, drawdown and sensitivity strategies where option bars exist", len(selected)],
    ],
    columns=["research_question", "primary_interpretation", "eligible_sessions"],
)
interpretation.to_csv(TABLE_ROOT / "fresh_holdout_interpretation_index.csv", index=False)
display(interpretation)
print("Saved fresh-holdout evidence under:", HOLDOUT_ROOT)


## What I do after this notebook runs

1. I preserve the complete output folder and do not overwrite the first successful run.
2. I report RQ1 and RQ3 classification metrics, but describe RQ2 using error rather than “accuracy”.
3. I report RQ4 coverage even when the combined rule selects no dates.
4. I report RQ5 as unavailable when candidate contracts or usable bars are missing; I do not change the filters to create trades.
5. I treat the LSTM comparison as exploratory.
6. Any later model changes require a new, later holdout period and cannot be judged on these same dates as if they were fresh.
